# 09 · All specialists — one run per document class

Every taxonomy class through the **real** graph: sorter → the matching
specialist → report → catalog → archive. Six live classes, five specialists
(contracts also covers MAUD `merger_agreement`; both share
`ContractExtraction`).

**What you'll see:** a per-class table of the specialist that dispatched, the
node path the router actually took, the stage that landed, and the extracted
keys. CUAD contracts and MAUD merger agreements ride the LangChain specialist;
the other four ride the legacy `agents.*` specialists (the same split the
production graph uses).

**Honesty label:** the graph, dispatch, schemas, bins, and catalog are REAL.
The LLMs are the test-suite mocks (`FakeLangChainLLM` for the LangChain path,
a scripted OpenAI client for the legacy specialists). Outputs are what the
pipeline produced under those mocks — OFFLINE, no API key.

Companion: `notebooks/pipeline_lab.py` (`CLASS_PACKS`, `run_all_classes`).


## Setup


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

import pipeline_lab as lab
lab.quiet_logs()


## The roster this notebook exercises

Each pack is a document + a canned classification + a canned extraction that
matches that class's Pydantic schema. `run_all_classes` scripts every legacy
specialist marker and runs one document per class.


In [2]:
packs = lab.CLASS_PACKS
print(f"{len(packs)} classes:")
for key, pack in packs.items():
    print(f"  {key:22s} {pack['specialist']:32s} path={pack['path']:9s} file={pack['filename']}")


7 classes:
  contract               contracts_specialist             path=langchain file=contract.txt
  corporate_record       corporate_records_specialist     path=legacy    file=bylaws.txt
  due_diligence          due_diligence_specialist         path=legacy    file=diligence.txt
  correspondence         correspondence_specialist        path=legacy    file=demand_letter.txt
  compliance_filing      compliance_specialist            path=legacy    file=form_8k.txt
  court_opinion          court_opinions_specialist        path=legacy    file=opinion.txt
  insurance_claim        insurance_claims_specialist      path=legacy    file=fnol.txt


## Run — one document per class

Each class gets its own `matter_id` so a shared field name (`effective_date`
on both contract and corporate_record) cannot look like a contradiction.
Same-class conflict is notebook 10; mixed matters are notebook 07.


In [3]:
env = lab.open_sandbox()
rows = lab.run_all_classes(env)
print(f"{'class':22s} {'stage':10s} {'keys':4s}  path")
print("-" * 88)
for row in rows:
    print(f"{row['doc_class']:22s} {str(row['stage']):10s} {len(row['extracted_keys']):4d}  {' → '.join(row['path'])}")
print()
print("all archived:" , all(r["stage"] == "archived" for r in rows))
print("all five classes dispatched:", [r["doc_class"] for r in rows])


class                  stage      keys  path
----------------------------------------------------------------------------------------
contract               archived     10  intake-document → classify-document → extract-fields → compile-report → write-catalog → archive-document
corporate_record       archived      7  intake-document → classify-document → extract-fields → compile-report → write-catalog → archive-document
due_diligence          archived      7  intake-document → classify-document → extract-fields → compile-report → write-catalog → archive-document
correspondence         archived      4  intake-document → classify-document → extract-fields → compile-report → write-catalog → archive-document
compliance_filing      archived      8  intake-document → classify-document → extract-fields → compile-report → write-catalog → archive-document
court_opinion          archived      4  intake-document → classify-document → extract-fields → compile-report → write-catalog → archive-docum

## What each specialist wrote

The extracted payload is the specialist's schema, not a generic bag of
fields — `claimed_amount: 0.0` on the insurance FNOL is a real value (see
notebook 10), not an empty extraction.


In [4]:
arts = lab.artifacts(env["base_dir"])
docs = (arts.get("catalog") or {}).get("documents")
if docs:
    cols = docs["columns"]
    i_type = cols.index("doc_type")
    i_data = cols.index("extracted_data")
    import json
    for rec in docs["rows"]:
        payload = rec[i_data]
        if isinstance(payload, str):
            try:
                payload = json.loads(payload)
            except Exception:
                payload = {}
        keys = sorted(k for k in (payload or {}) if not str(k).startswith("_"))
        print(f"{rec[i_type]:22s} {keys}")
lab.close_sandbox(env)


contract               ['contract_value', 'document_name', 'effective_date', 'governing_law', 'key_obligations', 'parties', 'reasoning', 'renewal_terms', 'term_length', 'termination_clauses']
corporate_record       ['effective_date', 'entity_name', 'filing_number', 'jurisdiction', 'key_provisions', 'record_type', 'signatories']
due_diligence          ['diligence_type', 'document_date', 'material_findings', 'outstanding_items', 'prepared_by', 'risk_flags', 'target_entity']
correspondence         ['communication_date', 'demand_amount', 'recipient', 'sender']
compliance_filing      ['due_date', 'entity_name', 'filing_date', 'filing_type', 'key_requirements', 'reference_number', 'regulatory_body', 'status']
court_opinion          ['case_name', 'court', 'date_decided', 'docket_number']
insurance_claim        ['adjuster', 'claim_number', 'claim_type', 'claimed_amount', 'coverage_determination', 'damages_description', 'date_filed', 'date_of_loss', 'denial_reasons', 'insured_party', 'insurer',

## Where to go next

- **10 edge_cases** — unknown type, missing CUAD subtype, $0 amounts, schema-invalid extract, same-class conflict → Boss
- **07 multi_document_matters** — several classes under one `matter_id`
- **11 huggingface_corpora** — the Lucius-Morningstar datasets these classes were published from
